Домашнее задание: Прогнозирование зарплаты разработчика на основе данных опроса Stack Overflow
Цель работы: Применить нейросети для решения реальной задачи — предсказания зарплаты (или её аналога, например, ConvertedCompYearly) на основе ответов разработчиков на ежегодный опрос. Вы научитесь обрабатывать сложные смешанные данные (числовые, категориальные, множественного выбора), строить нейросетевые модели и оценивать их качество.

Источник данных: Stack Overflow Annual Developer Survey. Используйте данные последнего доступного года (например, 2024). Файл с результатами обычно доступен для скачивания в формате CSV.

# Блок 0: Постановка задачи
В этом задании вам предстоит построить нейросеть, которая по ответам респондента (его профессиональный опыт, используемые технологии, образование, страна проживания и т.д.) будет предсказывать его годовую зарплату. Это задача регрессии.

Целевая переменная: ConvertedCompYearly (годовая зарплата в долларах США).

Основные шаги:

Загрузить и изучить данные.

Провести предобработку: отбор признаков, обработка пропусков, кодирование.

Построить и обучить нейросеть в PyTorch.

Оценить качество модели.

# Блок 1: Загрузка данных

Перейдите на сайт Stack Overflow Developer Survey. Найдите раздел с данными последнего опроса (например, «Download the Data»). Скачайте файл с результатами (обычно это архив .zip с файлом .csv). Распакуйте архив и загрузите данные в pandas.

Ваша задача: Написать код для загрузки данных. Укажите правильный путь к файлу.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
data = pd.read_csv('data/survey_results_public.csv')

In [3]:
data.describe()

,ResponseId,WorkExp,YearsCode,TechEndorse_1,TechEndorse_2,TechEndorse_3,TechEndorse_4,TechEndorse_5,TechEndorse_6,TechEndorse_7,...,SO_Actions_3,SO_Actions_4,SO_Actions_5,SO_Actions_6,SO_Actions_9,SO_Actions_7,SO_Actions_10,SO_Actions_15,ConvertedCompYearly,JobSat
count,49191.000000,42893.000000,43042.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,...,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,2.394700e+04,26670.000000
mean,24596.000000,13.367403,16.570861,7.867352,4.104211,4.110271,5.678193,4.119388,5.225990,6.477387,...,5.718355,4.561767,4.790861,5.199657,5.676314,4.984653,7.099505,10.079284,1.017615e+05,7.201950
std,14200.362883,10.800117,11.787610,2.397432,2.275821,2.329536,2.398084,2.437945,2.801045,2.331468,...,2.628016,3.070548,2.643177,2.563562,2.310659,2.490095,2.469394,1.940928,4.617569e+05,1.997245
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000e+00,0.000000
25%,12298.500000,5.000000,8.000000,7.000000,2.000000,2.000000,4.000000,2.000000,3.000000,5.000000,...,3.000000,1.000000,3.000000,3.000000,4.000000,3.000000,6.000000,10.000000,3.817100e+04,6.000000
50%,24596.000000,10.000000,14.000000,9.000000,4.000000,4.000000,6.000000,4.000000,5.000000,7.000000,...,6.000000,4.000000,5.000000,5.000000,6.000000,5.000000,8.000000,10.000000,7.532000e+04,8.000000
75%,36893.500000,20.000000,24.000000,9.000000,6.000000,6.000000,8.000000,6.000000,8.000000,8.000000,...,8.000000,7.000000,7.000000,7.000000,7.000000,7.000000,9.000000,10.000000,1.205960e+05,8.000000
max,49191.000000,100.000000,100.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,...,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,5.000000e+07,10.000000


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Columns: 172 entries, ResponseId to JobSat
dtypes: float64(52), int64(1), str(119)
memory usage: 64.6 MB


In [5]:
data.head().T

,0,1,2,3,4
ResponseId,1,2,3,4,5
MainBranch,I am a developer by profession,I am a developer by profession,I am a developer by profession,I am a developer by profession,I am a developer by profession
Age,25-34 years old,25-34 years old,35-44 years old,35-44 years old,35-44 years old
EdLevel,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Associate degree (A.A., A.S., etc.)","Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)"
Employment,Employed,Employed,"Independent contractor, freelancer, or self-em...",Employed,"Independent contractor, freelancer, or self-em..."
...,...,...,...,...,...
AIAgentExtWrite,NaN,NaN,NaN,NaN,NaN
AIHuman,When I don’t trust AI’s answers,When I don’t trust AI’s answers;When I want to...,When I don’t trust AI’s answers;When I want to...,When I don’t trust AI’s answers;When I want to...,When I don’t trust AI’s answers
AIOpen,"Troubleshooting, profiling, debugging",All skills. AI is a flop.,"Understand how things actually work, problem s...",NaN,"critical thinking, the skill to define the tas..."
ConvertedCompYearly,61256.0,104413.0,53061.0,36197.0,60000.0


In [6]:
data.shape

(49191, 172)

In [7]:
data.columns.tolist()

['ResponseId',
 'MainBranch',
 'Age',
 'EdLevel',
 'Employment',
 'EmploymentAddl',
 'WorkExp',
 'LearnCodeChoose',
 'LearnCode',
 'LearnCodeAI',
 'AILearnHow',
 'YearsCode',
 'DevType',
 'OrgSize',
 'ICorPM',
 'RemoteWork',
 'PurchaseInfluence',
 'TechEndorseIntro',
 'TechEndorse_1',
 'TechEndorse_2',
 'TechEndorse_3',
 'TechEndorse_4',
 'TechEndorse_5',
 'TechEndorse_6',
 'TechEndorse_7',
 'TechEndorse_8',
 'TechEndorse_9',
 'TechEndorse_13',
 'TechEndorse_13_TEXT',
 'TechOppose_1',
 'TechOppose_2',
 'TechOppose_3',
 'TechOppose_5',
 'TechOppose_7',
 'TechOppose_9',
 'TechOppose_11',
 'TechOppose_13',
 'TechOppose_16',
 'TechOppose_15',
 'TechOppose_15_TEXT',
 'Industry',
 'JobSatPoints_1',
 'JobSatPoints_2',
 'JobSatPoints_3',
 'JobSatPoints_4',
 'JobSatPoints_5',
 'JobSatPoints_6',
 'JobSatPoints_7',
 'JobSatPoints_8',
 'JobSatPoints_9',
 'JobSatPoints_10',
 'JobSatPoints_11',
 'JobSatPoints_13',
 'JobSatPoints_14',
 'JobSatPoints_15',
 'JobSatPoints_16',
 'JobSatPoints_15_TEXT',
 

# Блок 3: Предобработка данных
Данные опроса содержат множество столбцов разного типа. Для построения модели мы выберем ограниченный набор признаков. Вам нужно обработать пропуски, закодировать категориальные переменные и подготовить данные для нейросети.

In [8]:
missing = data.isnull().sum()
missing.sort_values(ascending=False)

AIAgentObsWrite      48927
SOTagsWant Entry     48761
SOTagsHaveEntry      48733
AIModelsWantEntry    48716
AIAgentOrchWrite     48713
                     ...  
EdLevel               1042
Employment             852
ResponseId               0
MainBranch               0
Age                      0
Length: 172, dtype: int64

In [9]:
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns
print(numeric_cols)
print(f'{len(numeric_cols)} - числовые признаки')

Index(['ResponseId', 'WorkExp', 'YearsCode', 'TechEndorse_1', 'TechEndorse_2',
       'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5', 'TechEndorse_6',
       'TechEndorse_7', 'TechEndorse_8', 'TechEndorse_9', 'TechEndorse_13',
       'TechOppose_1', 'TechOppose_2', 'TechOppose_3', 'TechOppose_5',
       'TechOppose_7', 'TechOppose_9', 'TechOppose_11', 'TechOppose_13',
       'TechOppose_16', 'TechOppose_15', 'JobSatPoints_1', 'JobSatPoints_2',
       'JobSatPoints_3', 'JobSatPoints_4', 'JobSatPoints_5', 'JobSatPoints_6',
       'JobSatPoints_7', 'JobSatPoints_8', 'JobSatPoints_9', 'JobSatPoints_10',
       'JobSatPoints_11', 'JobSatPoints_13', 'JobSatPoints_14',
       'JobSatPoints_15', 'JobSatPoints_16', 'ToolCountWork',
       'ToolCountPersonal', 'CompTotal', 'SO_Actions_1', 'SO_Actions_16',
       'SO_Actions_3', 'SO_Actions_4', 'SO_Actions_5', 'SO_Actions_6',
       'SO_Actions_9', 'SO_Actions_7', 'SO_Actions_10', 'SO_Actions_15',
       'ConvertedCompYearly', 'JobSat'],
     

In [10]:
category_cols = data.select_dtypes(include=['object']).columns
print(category_cols)
print(f'{len(category_cols)} - категориальные признаки')

Index(['MainBranch', 'Age', 'EdLevel', 'Employment', 'EmploymentAddl',
       'LearnCodeChoose', 'LearnCode', 'LearnCodeAI', 'AILearnHow', 'DevType',
       ...
       'AIAgentKnowledge', 'AIAgentKnowWrite', 'AIAgentOrchestration',
       'AIAgentOrchWrite', 'AIAgentObserveSecure', 'AIAgentObsWrite',
       'AIAgentExternal', 'AIAgentExtWrite', 'AIHuman', 'AIOpen'],
      dtype='str', length=119)
119 - категориальные признаки


In [11]:
data = data[data['ConvertedCompYearly'].notna()].copy()

In [12]:
data['ConvertedCompYearly'].isnull().sum()

np.int64(0)

In [13]:
# Выбор подходящих признаков для обучения модели + определение таргета
target = 'ConvertedCompYearly'
features = [
    'MainBranch',
    'Age',
    'EdLevel',
    'Employment',
    'WorkExp',
    'YearsCode',
    'DevType',
    'OrgSize',
    'RemoteWork',
    'Industry',
    'Country',
]
# Создание новой таблицы
data_clean = data[features + [target]].copy()

In [14]:
data_clean.info()

<class 'pandas.DataFrame'>
Index: 23947 entries, 0 to 49122
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   MainBranch           23947 non-null  str    
 1   Age                  23947 non-null  str    
 2   EdLevel              23930 non-null  str    
 3   Employment           23947 non-null  str    
 4   WorkExp              23469 non-null  float64
 5   YearsCode            23838 non-null  float64
 6   DevType              23947 non-null  str    
 7   OrgSize              21204 non-null  str    
 8   RemoteWork           21096 non-null  str    
 9   Industry             23033 non-null  str    
 10  Country              23947 non-null  str    
 11  ConvertedCompYearly  23947 non-null  float64
dtypes: float64(3), str(9)
memory usage: 2.4 MB


In [15]:
y = data_clean['ConvertedCompYearly']
data_clean  = data_clean.drop('ConvertedCompYearly', axis=1)
# обработка числовых признаков
data_clean['WorkExp'] = data_clean['WorkExp'].fillna(data_clean['WorkExp'].median())
data_clean['YearsCode'] = data_clean['YearsCode'].fillna(data_clean['YearsCode'].median())

In [16]:
data_clean.info()

<class 'pandas.DataFrame'>
Index: 23947 entries, 0 to 49122
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MainBranch  23947 non-null  str    
 1   Age         23947 non-null  str    
 2   EdLevel     23930 non-null  str    
 3   Employment  23947 non-null  str    
 4   WorkExp     23947 non-null  float64
 5   YearsCode   23947 non-null  float64
 6   DevType     23947 non-null  str    
 7   OrgSize     21204 non-null  str    
 8   RemoteWork  21096 non-null  str    
 9   Industry    23033 non-null  str    
 10  Country     23947 non-null  str    
dtypes: float64(2), str(9)
memory usage: 2.2 MB


# Блок 4: Создание нейросетевой модели в PyTorch
Теперь, когда данные готовы, перейдём к созданию нейросети. Определите архитектуру: входной слой (размерность равна количеству признаков после предобработки), несколько скрытых слоёв, выходной слой (1 нейрон для регрессии).

# Блок 5: Обучение модели
Напишите функцию обучения для регрессии. В качестве функции потерь используйте MSELoss (среднеквадратичная ошибка). Можно также считать MAE для оценки. Добавьте раннюю остановку и планировщик learning rate.

# Блок 6: Оценка модели
После обучения оцените модель на тестовой выборке. Поскольку это регрессия, используйте метрики:

MAE (Mean Absolute Error) — средняя абсолютная ошибка в долларах.

RMSE (Root Mean Squared Error) — корень из среднеквадратичной ошибки (чувствителен к выбросам).

R² (коэффициент детерминации) — доля объяснённой дисперсии.

Также постройте график истинных значений против предсказанных.

Ваша задача: Написать код для расчёта метрик и визуализации.

# Блок 7: Эксперименты и улучшения
Попробуйте улучшить модель, экспериментируя с разными аспектами:

Другие наборы признаков.

Разные архитектуры (глубина, ширина, активации).

Другие оптимизаторы (Adam, RMSprop, SGD).

Нормализация целевой переменной (например, логарифмирование зарплаты).

Борьба с выбросами (удаление зарплат выше 99-го процентиля).

Задание: Опишите, какие изменения вы внесли и как они повлияли на метрики. Приложите код и результаты.